# Lecture 05: Property-Based Testing: Some Examples

In this lecture:

* Anatomy of a property test with `Hypothesis`: `@settings` and `@given`
* Mechanics of failure and shrinking: how Hypothesis finds a minimal counterexample
* Case study: Run-Length Encoding (RLE): round-trip and structural invariants
* Testing the decoder against an independent generator: naive strategies, rejection sampling with `assume()`, constructive generation with `@st.composite`, and `.map()`

In [ ]:
# Setup and Imports
# %pip install hypothesis      # uncomment if Hypothesis is not installed
from hypothesis import given, settings, Verbosity, HealthCheck, strategies as st
import hypothesis

# Don't keep a database of past failures in a local .hypothesis/ folder:
# every run of this notebook starts from scratch
settings.register_profile("lecture", database=None)
settings.load_profile("lecture")

## Anatomy of a Property Test (`Hypothesis`)

In [ ]:
# Basic Property Test
# This test passes everywhere

@settings(max_examples=1000, verbosity=Verbosity.verbose)
#@settings(max_examples=100)
@given(st.integers())
def test_addition_identity(x: int):
    assert x + 0 == x
    assert x + 1 > x

# Execute the test explicitly in Jupyter
test_addition_identity()
print("Property passed successfully across automatically sampled integers!")

#### 1. Configuration (`@settings`)

* **`max_examples=1000`**: Overrides the default target of 100 iterations, instructing Hypothesis to sample and verify 1,000 distinct values.
* **`verbosity=Verbosity.verbose`**: Forces Hypothesis to print every attempted input trace to `stdout` in real-time, including shrinking steps.
* **Single Decorator Constraint**: Multiple settings must be combined into a single `@settings(...)` call rather than stacked.

#### 2. Strategy Injection (`@given`)

* **`st.integers()`**: Defines the strategy combinator $G$, generating values across the entire domain of integers ($\mathbb{Z}$).
* **Targeted Sampling**: Prioritizes known domain boundaries ($0, 1, -1$, powers-of-two boundaries, extreme limits) before uniform random sampling.

#### 3. Execution & Invariant Checking

* **Universal Quantification**: Represents $\forall x \in \mathbb{Z}. \; (x + 0 = x) \land (x + 1 > x)$.
* **Sequential Assertion Semantics**: Evaluates invariants in top-down order. The failure of *either* assertion immediately halts execution for input $x_i$ and triggers **binary shrinking** to locate the minimal counterexample.


## Mechanics of Failure & Shrinking

In [ ]:
# Intentional Failure to Observe Shrinking
@settings(max_examples=100, verbosity=Verbosity.verbose)
@given(st.integers())
def test_flawed_square_property(x: int):
    assert x**2 > x

# Running this will raise a Falsifying Example error
test_flawed_square_property()

#### 1. The Flawed Hypothesis

* **Property Under Test**: Asserts $x^2 > x$ across the domain of integers ($\mathbb{Z}$).
* **Domain Counterexamples**: Mathematically invalid for $x = 0$ ($0^2 \ngtr 0$) and $x = 1$ ($1^2 \ngtr 1$).

#### 2. Discovery & Interception

* **Sampling Phase**: Hypothesis samples values across $\mathbb{Z}$. It may initially hit a large counterexample (e.g., $x = 987,654$) or a boundary value.
* **Exception Catching**: When `assert x**2 > x` evaluates to `False`, Hypothesis intercepts the resulting `AssertionError` and halts further initial sampling.

#### 3. Delta/Binary Shrinking Algorithm

* **Minimization Strategy**: Hypothesis enters an automated minimization loop, applying binary search and byte-level reductions on the generator state to find the simplest input that still reproduces the failure.
* **Convergence**: Shrinks complex/large failing candidates down to the minimal algebraic counterexample **`x = 0`**.
* **Trace Isolation**: Produces a clean diagnostic output (`Falsifying example: x=0`), stripping away noise so you isolate root causes immediately.

In [ ]:
@settings(verbosity=Verbosity.verbose)
@given(st.integers())
def test_flawed_square_property_verbose(x: int):
    assert x**2 - 100 < x

# Run directly without try-except to let Hypothesis print its shrink sequence
test_flawed_square_property_verbose()

Mathematically, $x^2 - 100 < x$ is equivalent to $x^2 - x - 100 < 0$. Solving the quadratic equation $x^2 - x - 100 = 0$ gives roots at roughly $x \approx -9.51$ and $x \approx 10.51$.

* The inequality holds true for integers in the range $[-9, 10]$.
* The inequality **fails** for $x \ge 11$ and $x \le -10$.

Let's break down the exact operational steps Hypothesis took in one such trace (your run will differ in the details):

#### Initial Sampling & First Failure Discovery

* Hypothesis starts by probing boundary and random values:
* `x = 0` $\implies 0^2 - 100 < 0 \implies -100 < 0$ (**PASS**)
* `x = 1,391,879,151` $\implies$ huge $x^2 - 100 \nless x$ (**FAIL!**)


* Hypothesis intercepted the `AssertionError` for `1,391,879,151` and immediately entered its **shrinking phase** to find the smallest boundary counterexample near zero.


* `x = 36` (**FAIL**)
* `x = 18` (**FAIL**)
* `x = 16` (**FAIL**)
* `x = 12` (**FAIL**)
* `x = 10` $\implies 10^2 - 100 = 0 < 10$ (**PASS!**)
* `x = 11` $\implies 11^2 - 100 = 21 \nless 11$ (**FAIL!**)

At this point, Hypothesis knows $x = 10$ passes and $x = 11$ fails, establishing **$x = 11$** as the minimal positive counterexample.

#### Probing the Negative Domain

Next, Hypothesis shrinks along the negative integer axis to check if there is an even "smaller" counterexample in absolute terms or toward negative infinity:

* `x = -1, -9, -3, -5` (**PASS**)
* `x = -10` $\implies (-10)^2 - 100 = 0 \nless -10$ (**FAIL!**)
* `x = -11` (**FAIL**)

## Run-Length Encoding (RLE) Implementation

Run-Length Encoding is a simple data compression algorithm where consecutive identical elements are stored as a single data value and count.

In [ ]:
#Run-Length Encoding Implementation

import itertools

def rle_encode(data: str) -> list[tuple[str, int]]:
    """Encodes a string into a list of (character, count) tuples."""
    if not data:
        return []
    return [(char, len(list(group))) for char, group in itertools.groupby(data)]

def rle_decode(encoded: list[tuple[str, int]]) -> str:
    """Decodes a list of (character, count) tuples back into a string."""
    return "".join(char * count for char, count in encoded)

# Concrete Sanity Check
sample = "AAABBC"
encoded_sample = rle_encode(sample)
decoded_sample = rle_decode(encoded_sample)

print(f"Original : '{sample}'")
print(f"Encoded  : {encoded_sample}")
print(f"Decoded  : '{decoded_sample}'")

### Invariant Specification & Property Verification

When testing transformations like RLE, unit testing typically checks a few static strings (e.g., `"AAABBC"`). With PBT, we define two fundamental properties:

1. **Round-Trip Invariant ($f^{-1}(f(s)) = s$):** Decoding an encoded string must recover the exact original string across all possible string inputs ($\Sigma^*$).
2. **Structural Invariant:** The encoder must produce canonical runs—meaning no two adjacent runs in the output list can share the same character (e.g., `"AA"` shouldn't be encoded as `[('A', 1), ('A', 1)]`).

In [ ]:
# Property-Based Tests for RLE
from hypothesis import given, strategies as st

# Property 1: Round-Trip Invariant
@given(st.text())
def test_rle_roundtrip(s: str):
    assert rle_decode(rle_encode(s)) == s

# Property 2: Structural Invariant (No adjacent identical runs)
@given(st.text())
def test_rle_canonical_structure(s: str):
    encoded = rle_encode(s)
    for i in range(len(encoded) - 1):
        assert encoded[i][0] != encoded[i + 1][0]

# Execute both properties
test_rle_roundtrip()
test_rle_canonical_structure()
print("RLE Invariants verified successfully across automatically sampled strings!")

## Testing the Decoder Against an Independent Generator

### Naive Strategy & The Expected Breakdown

If we treat RLE payload data as an arbitrary sequence of `(character, integer)` pairs, we can build a naive strategy using nested Hypothesis primitives:

In [ ]:
# Strategy: arbitrary list of (char, int) tuples
naive_rle_strategy = st.lists(st.tuples(st.characters(), st.integers()))

@given(naive_rle_strategy)
def test_rle_decode_naive(encoded_data):
    # Invariant: Structural validity requires:
    # 1. Counts must be strictly positive (> 0)
    # 2. Adjacent characters must not be identical
    for char, count in encoded_data:
        assert count > 0, f"Invalid run length: {count}"
        
    for i in range(len(encoded_data) - 1):
        assert encoded_data[i][0] != encoded_data[i+1][0], "Adjacent runs share the same character"

# Running this will immediately trigger a falsifying counterexample
test_rle_decode_naive()

### The Limits of Naive Strategy Composition

* **The Naive Space**: `st.lists(st.tuples(st.characters(), st.integers()))` generates values from $\Sigma^* \times \mathbb{Z}$.
* **Domain Misalignment**: The valid RLE domain requires two strict invariants:
  1. $\forall (c, k) \in \text{data}, \quad k \in \mathbb{Z}^+$ (Strictly positive counts)
  2. $\forall i \in [0, |data|-2], \quad c_i \neq c_{i+1}$ (No adjacent duplicate keys)
* **Rejection Filtering vs. Construction**: Using `assume()` or `st.filter()` on naive strategies causes high rejection rates (Hypothesis fails a health check or gives up with an `Unsatisfiable` error). We must *constructively* generate valid instances using `@st.composite`.

### Quick Remedy: Rejection Sampling with `assume()`

The simplest way to filter out invalid generated data in Hypothesis is using `hypothesis.assume(...)`.

When `assume(condition)` evaluates to `False`, Hypothesis **discards the current test case** and tries to draw another example.

In [ ]:
# Filtering Naive Strategies with assume()
from hypothesis import given, assume, strategies as st

# Naive strategy
naive_rle_strategy = st.lists(st.tuples(st.characters(), st.integers()))
# suppress_health_check: we *know* this strategy filters out a lot (see below)
# (verbose output of 1000 examples is several MB: add verbosity=Verbosity.verbose to watch the rejections)
@settings(max_examples=1000, suppress_health_check=[HealthCheck.filter_too_much])
@given(naive_rle_strategy)
def test_rle_decode_with_assume(encoded_data):
    # Pre-condition 1: Counts must be strictly positive
    for char, count in encoded_data:
        assume(0 < count <= 10_000)
        
    # Pre-condition 2: Adjacent characters must not be identical
    for i in range(len(encoded_data) - 1):
        assume(encoded_data[i][0] != encoded_data[i+1][0])
        
    # Test property on filtered data
    decoded = rle_decode(encoded_data)
    assert len(decoded) == sum(count for _, count in encoded_data)

test_rle_decode_with_assume()

#### Finding a needle in a haystack

* It works! But if you look closely or run it in verbose mode, Hypothesis throws away a huge percentage of generated inputs because finding valid RLE payloads randomly out of `st.integers()` (which generates negative numbers, zeroes, etc.) is like finding a needle in a haystack.
* Hypothesis even warns us: without `suppress_health_check=[HealthCheck.filter_too_much]`, the test fails with a `FailedHealthCheck` (*"It looks like your strategy is filtering out a lot of data"*).
* If conditions are too strict, Hypothesis will eventually give up and raise an `Unsatisfiable` error: *"Hypothesis could not find enough valid examples pass assume()!"*

### Constructive Generation (`@st.composite`)


In [ ]:
# Custom Generator for Valid RLE Encoded Data
from hypothesis import given, strategies as st

@st.composite
def valid_rle_data(draw):
    """
    Constructively generates valid RLE data:
    1. Draws characters and removes adjacent duplicates.
    2. Assigns strictly positive counts (k >= 1) directly.
    """
    # Step 1: Draw raw characters
    raw_chars = draw(st.lists(st.characters(), min_size=0))
    
    # Step 2: Ensure adjacent characters are actually different
    filtered_chars = []
    for c in raw_chars:
        if not filtered_chars or filtered_chars[-1] != c:
            filtered_chars.append(c)
            
    # Step 3: Draw positive counts for each run directly
    result = []
    for c in filtered_chars:
        count = draw(st.integers(min_value=1, max_value=50))
        result.append((c, count))
        
    return result

@given(valid_rle_data())
def test_rle_decode_properties(encoded_data):
    decoded = rle_decode(encoded_data)
    
    # Structural Invariant: Length of decoded string matches sum of counts
    assert len(decoded) == sum(count for _, count in encoded_data)
    
    # Round-Trip Invariant: Re-encoding yields original canonical data
    assert rle_encode(decoded) == encoded_data

test_rle_decode_properties()
print("Custom strategy test passed efficiently!")

### Filtering vs. Constructive Generation

1. **Rejection Filtering (`assume`)**: 
   * Post-hoc filtering: generates unconstrained inputs and discards invalid ones via `assume(condition)`.
   * **Drawback**: High rejection rates. If the valid input space is sparse, Hypothesis aborts with an `Unsatisfiable` error.

2. **Constructive Generation (`@st.composite`)**:
   * Uses `draw()` to build valid domain structures directly.
   * **Benefit**: Guarantees 100% of generated samples satisfy domain preconditions without discarding any test runs.

In [ ]:
# The ultimate declarative shortcut (assumes correctness of rle_encode)

valid_rle_from_string = st.text().map(rle_encode)

@settings(max_examples=1000, verbosity=Verbosity.verbose)
@given(valid_rle_from_string)
def test_rle_decode_mapped(encoded_data):
    decoded = rle_decode(encoded_data)
    assert rle_encode(decoded) == encoded_data

test_rle_decode_mapped()